# Transfer Learning No-GAN Model Comparison 5-Fold

Notebook ini membandingkan hasil transfer learning tanpa GAN dengan 5-fold cross-validation, mengikuti format output `transfer_learning_sucess_crossval.ipynb`.

Model lokal mengikuti arsitektur `pretrained_model_comparison.ipynb`:

- `cnn_lstm` tanpa attention
- `cnn1d`
- `GRU`
- `lstm`

Model eksternal disiapkan opsional:

- HuBERT ECG official pretrained lokal dari `model_pretrain_comparison/hubert_ecg`, melalui `transformers`/`trust_remote_code`
- ECG-FM official foundation model, melalui `fairseq_signals` dan checkpoint `wanglab/ecg-fm`

Fokus eksperimen: **full fine-tuning only**. Tidak ada frozen backbone, partial fine-tuning, atau without-finetune baseline.


In [11]:
import os
import json
import warnings
import random
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_curve,
)

sns.set_theme(style='whitegrid')

## 1. Configuration

In [12]:
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'notebook' else Path('.').resolve()
DATA_DIR = PROJECT_ROOT / 'data_ready_ptb'
PRETRAIN_DIR = PROJECT_ROOT / 'model_pretrain_comparison'
OUTPUT_ROOT = PROJECT_ROOT / 'crossval_comparison'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

class_names = ['NORM', 'IMI', 'AMI', 'LMI']
lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

CONFIG = {
    'seed': 42,
    'n_splits': 5,
    'batch_size': 32,
    'epochs': 100,
    'learning_rate': 3e-4,
    'recurrent_learning_rate': 1e-3,
    'external_learning_rate': 1e-5,
    'weight_decay': 5e-4,
    'recurrent_weight_decay': 1e-7,
    'gradient_clip_norm': 1.0,
    'use_weighted_sampler': True,
    'use_class_weight': True,
    'selection_metric': 'macro_f1',
    'use_early_stopping': False,
    'patience': None,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    # Isi model id/path ini jika checkpoint official tersedia di environment.
    'enable_hubert_ecg': True,
    'hubert_ecg_model_id_or_path': str(PRETRAIN_DIR / 'hubert_ecg'),
    'enable_ecg_fm': True,
    'ecg_fm_checkpoint_path': str(PRETRAIN_DIR / 'ecg_fm' / 'mimic_iv_ecg_physionet_pretrained.pt'),
    'ecg_fm_pooling': 'mean',
}

device = torch.device(CONFIG['device'])
print(json.dumps(CONFIG, indent=2))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)
print('PRETRAIN_DIR:', PRETRAIN_DIR)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


{
  "seed": 42,
  "n_splits": 5,
  "batch_size": 32,
  "epochs": 100,
  "learning_rate": 0.0003,
  "recurrent_learning_rate": 0.001,
  "external_learning_rate": 1e-05,
  "weight_decay": 0.0005,
  "recurrent_weight_decay": 1e-07,
  "gradient_clip_norm": 1.0,
  "use_weighted_sampler": true,
  "use_class_weight": true,
  "selection_metric": "macro_f1",
  "use_early_stopping": false,
  "patience": null,
  "device": "cuda",
  "enable_hubert_ecg": true,
  "hubert_ecg_model_id_or_path": "/home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/model_pretrain_comparison/hubert_ecg",
  "enable_ecg_fm": true,
  "ecg_fm_checkpoint_path": "/home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/model_pretrain_comparison/ecg_fm/mimic_iv_ecg_physionet_pretrained.pt",
  "ecg_fm_pooling": "mean"
}
PROJECT_ROOT: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification
DATA_DIR: /home/nugee/code-program/code-thesis/hibah/myocardial-in

In [13]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(CONFIG['seed'])

## 2. Load Data Ready PTB No-GAN

In [14]:
def load_split(data_dir, split):
    x = np.load(data_dir / f'x_{split}.npy').astype(np.float32)
    y = np.load(data_dir / f'y_{split}.npy')
    if y.ndim > 1 and y.shape[1] > 1:
        y = y.argmax(axis=1).astype(np.int64)
    else:
        y = y.reshape(-1).astype(np.int64)
    return x, y

def validate_ecg_array(x, name):
    if x.ndim != 3:
        raise ValueError(f'{name}: expected [N, length, leads], got {x.shape}')
    if x.shape[2] != len(lead_names):
        raise ValueError(f'{name}: expected 12 leads, got {x.shape[2]}')
    if not np.isfinite(x).all():
        raise ValueError(f'{name}: contains NaN or Inf')

x_train, y_train = load_split(DATA_DIR, 'train')
x_val, y_val = load_split(DATA_DIR, 'val')
x_test, y_test = load_split(DATA_DIR, 'test')
for split_name, x in [('train', x_train), ('val', x_val), ('test', x_test)]:
    validate_ecg_array(x, split_name)

x_cv = np.concatenate([x_train, x_val], axis=0).astype(np.float32)
y_cv = np.concatenate([y_train, y_val], axis=0).astype(np.int64)
x_test = x_test.astype(np.float32)
y_test = y_test.astype(np.int64)

print('Combined CV data:', x_cv.shape, np.bincount(y_cv, minlength=len(class_names)))
print('Holdout test    :', x_test.shape, np.bincount(y_test, minlength=len(class_names)))
for idx, name in enumerate(class_names):
    print(f'{name:4} | CV: {int((y_cv == idx).sum()):4d} | Test: {int((y_test == idx).sum()):4d}')

Combined CV data: (163, 1000, 12) [64 64 33  2]
Holdout test    : (42, 1000, 12) [16 16  9  1]
NORM | CV:   64 | Test:   16
IMI  | CV:   64 | Test:   16
AMI  | CV:   33 | Test:    9
LMI  | CV:    2 | Test:    1


## 3. Dataset, Normalization, and Loaders

In [15]:
class PerLeadZScore:
    def __init__(self, eps=1e-6):
        self.eps = eps
        self.mean_ = None
        self.std_ = None

    def fit(self, x):
        self.mean_ = x.mean(axis=(0, 1), keepdims=True)
        self.std_ = np.maximum(x.std(axis=(0, 1), keepdims=True), self.eps)
        return self

    def transform(self, x):
        if self.mean_ is None or self.std_ is None:
            raise RuntimeError('Normalizer must be fit on the current train fold first.')
        return ((x - self.mean_) / self.std_).astype(np.float32)

class ECGDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.as_tensor(x, dtype=torch.float32)
        self.y = torch.as_tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

def make_loaders_for_fold(train_idx, val_idx, batch_size):
    normalizer = PerLeadZScore().fit(x_cv[train_idx])
    x_fold_train = normalizer.transform(x_cv[train_idx])
    x_fold_val = normalizer.transform(x_cv[val_idx])
    x_fold_test = normalizer.transform(x_test)

    y_fold_train = y_cv[train_idx]
    y_fold_val = y_cv[val_idx]

    train_ds = ECGDataset(x_fold_train, y_fold_train)
    val_ds = ECGDataset(x_fold_val, y_fold_val)
    test_ds = ECGDataset(x_fold_test, y_test)

    if CONFIG['use_weighted_sampler']:
        counts = np.bincount(y_fold_train, minlength=len(class_names))
        weights = 1.0 / np.maximum(counts, 1)
        sample_weights = weights[y_fold_train]
        sampler = WeightedRandomSampler(
            weights=torch.tensor(sample_weights, dtype=torch.double),
            num_samples=len(sample_weights),
            replacement=True,
        )
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, normalizer

## 4. Model Architectures from Pretrained Model Comparison

In [16]:
class ECG1DNet(nn.Module):
    input_format = 'blc'
    def __init__(self, in_channels=12, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(16),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.6),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B,L,C] -> [B,C,L]
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

class LSTM_Only_Multi(nn.Module):
    input_format = 'blc'
    def __init__(self, input_size=12, num_classes=4, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        feat = out[:, -1, :]
        return self.head(feat)

class GRU_Only_Multi(nn.Module):
    input_format = 'blc'
    def __init__(self, input_size=12, num_classes=4, hidden_size=128, num_layers=3, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        out, _ = self.gru(x)
        feat = out[:, -1, :]
        return self.head(feat)

class CNN_LSTM_Baseline(nn.Module):
    input_format = 'blc'
    def __init__(self, in_channels=12, num_classes=4, lstm_hidden=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, 3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(32),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(64),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1),
            nn.ReLU(), nn.BatchNorm1d(128),
        )
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=False,
            dropout=0.35,
        )
        self.head = nn.Sequential(
            nn.Dropout(0.55),
            nn.Linear(lstm_hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.55),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B,L,C] -> [B,C,L]
        x = self.conv(x)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        feat = out[:, -1, :]
        return self.head(feat)

## 5. Optional Official Pretrained Models

In [17]:
import importlib

class HuBERT_ECG_Classifier(nn.Module):
    def __init__(self, hubert_name, num_classes=4, dropout=0.3):
        super().__init__()
        from transformers import AutoModel
        self.backbone = AutoModel.from_pretrained(hubert_name, trust_remote_code=True)
        hidden_size = int(self.backbone.config.hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # Reference notebook policy: x [B,L,12] -> [B,12,L] -> mean across leads -> [B,L].
        x_bcl = x.permute(0, 2, 1)
        x_mean = x_bcl.mean(dim=1)
        outputs = self.backbone(input_values=x_mean)
        hs = outputs.last_hidden_state
        pooled = hs.mean(dim=1)
        return self.fc(self.dropout(pooled))


def build_hubert_ecg_model(num_classes=4):
    model_id = CONFIG.get('hubert_ecg_model_id_or_path')
    if not CONFIG.get('enable_hubert_ecg', False) or not model_id:
        return None, {'available': False, 'reason': 'hubert_ecg_model_id_or_path is not configured'}
    try:
        model = HuBERT_ECG_Classifier(model_id, num_classes=num_classes, dropout=0.3)
        return model, {'available': True, 'loaded': True, 'reason': f'loaded {model_id}'}
    except Exception as exc:
        return None, {'available': False, 'loaded': False, 'reason': repr(exc)}


def get_logits(out):
    if isinstance(out, torch.Tensor):
        return out
    if isinstance(out, (tuple, list)) and len(out) > 0 and isinstance(out[0], torch.Tensor):
        return out[0]
    if isinstance(out, dict):
        for key in ['logits', 'pred', 'output', 'y_hat']:
            if key in out and isinstance(out[key], torch.Tensor):
                return out[key]
        for value in out.values():
            if isinstance(value, torch.Tensor):
                return value
    raise TypeError(f'Cannot extract logits from output type: {type(out)}')


class ECGFMClassifier(nn.Module):
    def __init__(self, checkpoint_path, num_classes=4, dropout=0.3, pooling='mean'):
        super().__init__()
        from fairseq_signals.models import build_model_from_checkpoint

        self.backbone = build_model_from_checkpoint(checkpoint_path=str(checkpoint_path))
        self.pooling = pooling
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.LazyLinear(num_classes)

    def forward(self, x):
        # Project loader gives [B,L,12]. ECG-FM/fairseq_signals expects [B,12,L].
        x_bcl = x.permute(0, 2, 1).contiguous()
        out = self.backbone(source=x_bcl)
        features = out.get('features') if isinstance(out, dict) else None
        if features is None:
            raise TypeError('ECG-FM output does not contain `features` tensor')
        if features.dim() != 3:
            raise ValueError(f'Unexpected ECG-FM feature shape: {tuple(features.shape)}')
        if self.pooling == 'max':
            pooled = features.max(dim=1).values
        else:
            pooled = features.mean(dim=1)
        return self.fc(self.dropout(pooled))


def build_ecg_fm_model(num_classes=4):
    if not CONFIG.get('enable_ecg_fm', False):
        return None, {'available': False, 'loaded': False, 'reason': 'enable_ecg_fm is False'}
    checkpoint_path = Path(CONFIG.get('ecg_fm_checkpoint_path', ''))
    if not checkpoint_path.exists():
        return None, {'available': False, 'loaded': False, 'reason': f'missing ECG-FM checkpoint: {checkpoint_path}'}
    try:
        model = ECGFMClassifier(
            checkpoint_path=checkpoint_path,
            num_classes=num_classes,
            dropout=0.3,
            pooling=CONFIG.get('ecg_fm_pooling', 'mean'),
        )
        return model, {'available': True, 'loaded': True, 'reason': f'loaded ECG-FM checkpoint: {checkpoint_path}'}
    except Exception as exc:
        return None, {'available': False, 'loaded': False, 'reason': repr(exc)}


## 6. Model Registry and Pretrained Loading

In [18]:
def safe_torch_load(path):
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')
    except Exception:
        return torch.load(path, map_location='cpu', weights_only=False)

def load_state_dict_flexible(model, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        return {'loaded': False, 'reason': f'missing checkpoint: {checkpoint_path}', 'missing_keys': [], 'unexpected_keys': []}
    ckpt = safe_torch_load(checkpoint_path)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        state_dict = ckpt['model_state_dict']
    elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
        state_dict = ckpt['state_dict']
    else:
        state_dict = ckpt
    result = model.load_state_dict(state_dict, strict=False)
    return {
        'loaded': True,
        'reason': str(checkpoint_path),
        'missing_keys': list(result.missing_keys),
        'unexpected_keys': list(result.unexpected_keys),
    }

def build_local_model(model_name):
    if model_name == 'cnn1d':
        return ECG1DNet(in_channels=len(lead_names), num_classes=len(class_names))
    if model_name == 'cnn_lstm':
        return CNN_LSTM_Baseline(in_channels=len(lead_names), num_classes=len(class_names), lstm_hidden=64)
    if model_name == 'lstm':
        return LSTM_Only_Multi(input_size=len(lead_names), num_classes=len(class_names), hidden_size=128, num_layers=3, dropout=0.2)
    if model_name == 'GRU':
        return GRU_Only_Multi(input_size=len(lead_names), num_classes=len(class_names), hidden_size=128, num_layers=3, dropout=0.2)
    raise ValueError(model_name)

MODEL_CONFIGS = [
    {'model_name': 'cnn_lstm', 'kind': 'local', 'pretrain_path': PRETRAIN_DIR / 'cnn_lstm.pth', 'learning_rate': CONFIG['learning_rate'], 'weight_decay': CONFIG['weight_decay']},
    {'model_name': 'cnn1d', 'kind': 'local', 'pretrain_path': PRETRAIN_DIR / 'cnn1d.pth', 'learning_rate': CONFIG['learning_rate'], 'weight_decay': CONFIG['weight_decay']},
    {'model_name': 'GRU', 'kind': 'local', 'pretrain_path': PRETRAIN_DIR / 'GRU.pth', 'learning_rate': CONFIG['recurrent_learning_rate'], 'weight_decay': CONFIG['recurrent_weight_decay']},
    {'model_name': 'lstm', 'kind': 'local', 'pretrain_path': PRETRAIN_DIR / 'lstm.pth', 'learning_rate': CONFIG['recurrent_learning_rate'], 'weight_decay': CONFIG['recurrent_weight_decay']},
    {'model_name': 'hubert_ecg', 'kind': 'hubert', 'learning_rate': CONFIG['external_learning_rate'], 'weight_decay': CONFIG['weight_decay']},
    {'model_name': 'ecg_fm', 'kind': 'ecg_fm', 'learning_rate': CONFIG['external_learning_rate'], 'weight_decay': CONFIG['weight_decay']},
]

def instantiate_model(model_cfg):
    if model_cfg['kind'] == 'local':
        model = build_local_model(model_cfg['model_name'])
        load_info = load_state_dict_flexible(model, model_cfg['pretrain_path'])
        return model, load_info
    if model_cfg['kind'] == 'hubert':
        return build_hubert_ecg_model(num_classes=len(class_names))
    if model_cfg['kind'] == 'ecg_fm':
        return build_ecg_fm_model(num_classes=len(class_names))
    raise ValueError(model_cfg['kind'])

availability_rows = []
for cfg in MODEL_CONFIGS:
    try:
        probe_model, info = instantiate_model(cfg)
        available = probe_model is not None and info.get('available', True) is not False
        availability_rows.append({'model_name': cfg['model_name'], 'kind': cfg['kind'], 'available': available, 'reason': info.get('reason', ''), 'pretrain_loaded': info.get('loaded', available)})
        del probe_model
    except Exception as exc:
        availability_rows.append({'model_name': cfg['model_name'], 'kind': cfg['kind'], 'available': False, 'reason': repr(exc), 'pretrain_loaded': False})
model_availability_df = pd.DataFrame(availability_rows)
display(model_availability_df)
model_availability_df.to_csv(OUTPUT_ROOT / 'localization_model_availability.csv', index=False)

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

,model_name,kind,available,reason,pretrain_loaded
0,cnn_lstm,local,True,/home/nugee/code-program/code-thesis/hibah/myo...,True
1,cnn1d,local,True,/home/nugee/code-program/code-thesis/hibah/myo...,True
2,GRU,local,True,/home/nugee/code-program/code-thesis/hibah/myo...,True
3,lstm,local,True,/home/nugee/code-program/code-thesis/hibah/myo...,True
4,hubert_ecg,hubert,True,loaded /home/nugee/code-program/code-thesis/hi...,True
5,ecg_fm,ecg_fm,True,loaded ECG-FM checkpoint: /home/nugee/code-pro...,True


## 7. Training and Evaluation Helpers

In [19]:
def count_trainable_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def make_criterion(y_train_fold):
    if not CONFIG['use_class_weight']:
        return nn.CrossEntropyLoss()
    counts = np.bincount(y_train_fold, minlength=len(class_names))
    class_weights = counts.sum() / np.maximum(counts, 1)
    class_weights = class_weights / class_weights.mean()
    return nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = get_logits(model(xb))
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip_norm'])
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_count += xb.size(0)
    return total_loss / max(total_count, 1), total_correct / max(total_count, 1)

def safe_balanced_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    support = cm.sum(axis=1)
    present = support > 0
    recalls = np.divide(np.diag(cm), support, out=np.zeros_like(support, dtype=float), where=support > 0)
    return float(recalls[present].mean()) if np.any(present) else np.nan

def evaluate_model(model, loader, criterion=None):
    model.eval()
    total_loss, total_count = 0.0, 0
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = get_logits(model(xb))
            if criterion is not None:
                loss = criterion(logits, yb)
                total_loss += loss.item() * xb.size(0)
                total_count += xb.size(0)
            prob = torch.softmax(logits, dim=1)
            y_true.append(yb.cpu().numpy())
            y_pred.append(prob.argmax(dim=1).cpu().numpy())
            y_prob.append(prob.cpu().numpy())
    y_true = np.concatenate(y_true)
    y_pred = np.concatenate(y_pred)
    y_prob = np.vstack(y_prob)
    summary = {
        'loss': total_loss / max(total_count, 1) if criterion is not None else np.nan,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': safe_balanced_accuracy(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', zero_division=0),
    }
    return summary, y_true, y_pred, y_prob

def plot_training_curve(history, title, save_path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    axes[0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0].plot(df['epoch'], df['val_loss'], label='Val', linestyle='--')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(alpha=0.3)
    axes[0].legend()
    axes[1].plot(df['epoch'], df['train_acc'], label='Train')
    axes[1].plot(df['epoch'], df['val_acc'], label='Val', linestyle='--')
    axes[1].plot(df['epoch'], df['val_macro_f1'], label='Val Macro F1', linestyle=':')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].grid(alpha=0.3)
    axes[1].legend()
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_confusion_matrix(y_true, y_pred, title, save_path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    plt.figure(figsize=(7, 6), dpi=150)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return cm

def plot_roc_pr_curves(y_true, y_prob, title_prefix, save_path):
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
    for i, name in enumerate(class_names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
        roc_auc = auc(fpr, tpr)
        precision, recall, _ = precision_recall_curve(y_bin[:, i], y_prob[:, i])
        ap = average_precision_score(y_bin[:, i], y_prob[:, i])
        axes[0].plot(fpr, tpr, label=f'{name} AUC={roc_auc:.3f}')
        axes[1].plot(recall, precision, label=f'{name} AP={ap:.3f}')
    axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', linewidth=1)
    axes[0].set_title(f'{title_prefix} ROC')
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    axes[1].set_title(f'{title_prefix} Precision-Recall')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def classification_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        labels=list(range(len(class_names))),
        digits=4,
        zero_division=0,
        output_dict=True,
    )
    return pd.DataFrame(report).T


## 8. Run 5-Fold Cross-Validation Full Fine-Tuning

In [20]:
def make_rare_class_tolerant_folds(y, n_splits=5, seed=42):
    rng = np.random.default_rng(seed)
    fold_indices = [[] for _ in range(n_splits)]
    class_counts = np.bincount(y, minlength=len(class_names))
    if class_counts.min() < n_splits:
        warnings.warn(
            f'Minimum class count in CV data is {class_counts.min()}, smaller than n_splits={n_splits}. '
            'Using rare-class tolerant round-robin folds; some validation folds may not contain every class.',
            RuntimeWarning,
        )
    for class_idx in range(len(class_names)):
        idx = np.where(y == class_idx)[0]
        rng.shuffle(idx)
        for pos, sample_idx in enumerate(idx):
            fold_indices[pos % n_splits].append(int(sample_idx))
    all_idx = np.arange(len(y))
    folds = []
    for fold_idx in range(n_splits):
        val_idx = np.array(sorted(fold_indices[fold_idx]), dtype=int)
        train_idx = np.setdiff1d(all_idx, val_idx, assume_unique=False)
        folds.append((train_idx, val_idx))
    return folds

folds = make_rare_class_tolerant_folds(y_cv, CONFIG['n_splits'], CONFIG['seed'])
fold_distribution = []
for fold, (_, val_idx) in enumerate(folds, start=1):
    counts = np.bincount(y_cv[val_idx], minlength=len(class_names))
    fold_distribution.append({'fold': fold, **{name: int(counts[i]) for i, name in enumerate(class_names)}})
fold_distribution_df = pd.DataFrame(fold_distribution)
display(fold_distribution_df)
fold_distribution_df.to_csv(OUTPUT_ROOT / 'localization_no_gan_fold_distribution.csv', index=False)

all_fold_summaries = []
all_histories = {}

for model_cfg in MODEL_CONFIGS:
    model_name = model_cfg['model_name']
    model_dir = OUTPUT_ROOT / f'localization_{model_name}'
    (model_dir / 'fold_models').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_histories').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_plots').mkdir(parents=True, exist_ok=True)
    (model_dir / 'fold_reports').mkdir(parents=True, exist_ok=True)

    print(f"\n{'=' * 90}\nMODEL: {model_name}\nOUTPUT: {model_dir}\n{'=' * 90}")

    for fold, (train_idx, val_idx) in enumerate(folds, start=1):
        set_seed(CONFIG['seed'] + fold)
        fold_dir = model_dir / 'fold_plots' / f'fold{fold}'
        fold_dir.mkdir(parents=True, exist_ok=True)

        train_loader, val_loader, test_loader, normalizer = make_loaders_for_fold(train_idx, val_idx, CONFIG['batch_size'])
        model, load_info = instantiate_model(model_cfg)
        if model is None:
            skip_row = {
                'model_name': model_name,
                'fold': fold,
                'status': 'skipped',
                'skip_reason': load_info.get('reason', 'model unavailable'),
            }
            all_fold_summaries.append(skip_row)
            print(f"Fold {fold} skipped: {skip_row['skip_reason']}")
            continue

        model = model.to(device)
        # Materialize LazyLinear for optional transformer wrappers before optimizer creation.
        try:
            with torch.no_grad():
                dummy_x = torch.zeros(2, x_cv.shape[1], x_cv.shape[2], dtype=torch.float32, device=device)
                _ = model(dummy_x)
        except Exception as exc:
            skip_row = {
                'model_name': model_name,
                'fold': fold,
                'status': 'skipped',
                'skip_reason': f'forward probe failed: {repr(exc)}',
            }
            all_fold_summaries.append(skip_row)
            print(f"Fold {fold} skipped: {skip_row['skip_reason']}")
            del model
            continue

        for p in model.parameters():
            p.requires_grad = True
        total_params, trainable_params = count_trainable_parameters(model)
        criterion = make_criterion(y_cv[train_idx])
        optimizer = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=model_cfg.get('learning_rate', CONFIG['learning_rate']),
            weight_decay=model_cfg.get('weight_decay', CONFIG['weight_decay']),
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

        best_score = -np.inf
        best_epoch = 0
        best_state = None
        epochs_without_improvement = 0
        history = []

        print(
            f"Fold {fold}/{CONFIG['n_splits']} | train={len(train_idx)} val={len(val_idx)} "
            f"| pretrained={load_info.get('loaded', load_info.get('available', False))} "
            f"| trainable={trainable_params:,}/{total_params:,}"
        )

        for epoch in range(1, CONFIG['epochs'] + 1):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
            val_summary, _, _, _ = evaluate_model(model, val_loader, criterion)
            selection_score = val_summary[CONFIG['selection_metric']]
            scheduler.step(selection_score)

            row = {
                'model_name': model_name,
                'fold': fold,
                'epoch': epoch,
                'train_loss': train_loss,
                'train_acc': train_acc,
                'val_loss': val_summary['loss'],
                'val_acc': val_summary['accuracy'],
                'val_balanced_accuracy': val_summary['balanced_accuracy'],
                'val_macro_f1': val_summary['macro_f1'],
                'val_weighted_f1': val_summary['weighted_f1'],
                'learning_rate': optimizer.param_groups[0]['lr'],
            }
            history.append(row)

            if selection_score > best_score:
                best_score = selection_score
                best_epoch = epoch
                best_state = deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epoch == 1 or epoch % 5 == 0 or epoch == CONFIG['epochs']:
                print(
                    f"Epoch {epoch:03d} | train_loss={train_loss:.4f} val_loss={val_summary['loss']:.4f} "
                    f"val_acc={val_summary['accuracy']:.4f} val_macro_f1={val_summary['macro_f1']:.4f}"
                )

            if CONFIG.get('use_early_stopping', False) and epochs_without_improvement >= CONFIG['patience']:
                print(f"Early stopping at epoch {epoch}; best epoch {best_epoch} ({CONFIG['selection_metric']}={best_score:.4f})")
                break

        if best_state is None:
            warnings.warn(f'No best state for {model_name} fold {fold}; skipping evaluation.', RuntimeWarning)
            continue

        model.load_state_dict(best_state)
        hist_df = pd.DataFrame(history)
        hist_path = model_dir / 'fold_histories' / f'fold{fold}_history.csv'
        hist_df.to_csv(hist_path, index=False)
        all_histories[f'{model_name}_fold{fold}'] = hist_df

        ckpt_path = model_dir / 'fold_models' / f'fold{fold}_best_epoch{best_epoch}_{CONFIG["selection_metric"]}{best_score:.4f}.pth'
        torch.save({
            'model_state_dict': best_state,
            'model_name': model_name,
            'fold': fold,
            'best_epoch': best_epoch,
            'best_score': float(best_score),
            'class_names': class_names,
            'lead_names': lead_names,
            'normalizer_mean': normalizer.mean_,
            'normalizer_std': normalizer.std_,
            'config': CONFIG,
            'model_config': {k: str(v) if isinstance(v, Path) else v for k, v in model_cfg.items()},
            'load_info': load_info,
            'finetuning_mode': 'full',
        }, ckpt_path)

        val_summary, y_val_true, y_val_pred, y_val_prob = evaluate_model(model, val_loader, criterion)
        test_summary, y_test_true, y_test_pred, y_test_prob = evaluate_model(model, test_loader, criterion)
        val_cm = plot_confusion_matrix(y_val_true, y_val_pred, f'{model_name} Fold {fold} Validation', fold_dir / 'confusion_matrix_validation.png')
        test_cm = plot_confusion_matrix(y_test_true, y_test_pred, f'{model_name} Fold {fold} Test', fold_dir / 'confusion_matrix_test.png')
        plot_training_curve(hist_df, f'{model_name} - Fold {fold}', fold_dir / 'training_curve.png')
        plot_roc_pr_curves(y_val_true, y_val_prob, f'{model_name} Fold {fold} Validation', fold_dir / 'roc_pr_validation.png')
        plot_roc_pr_curves(y_test_true, y_test_prob, f'{model_name} Fold {fold} Test', fold_dir / 'roc_pr_test.png')

        classification_report_df(y_val_true, y_val_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_classification_report.csv')
        classification_report_df(y_test_true, y_test_pred).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_classification_report.csv')
        pd.DataFrame(val_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_validation_confusion_matrix.csv')
        pd.DataFrame(test_cm, index=class_names, columns=class_names).to_csv(model_dir / 'fold_reports' / f'fold{fold}_test_confusion_matrix.csv')

        summary_row = {
            'model_name': model_name,
            'fold': fold,
            'status': 'completed',
            'finetuning_mode': 'full',
            'pretrain_loaded': load_info.get('loaded', load_info.get('available', False)),
            'pretrain_reason': load_info.get('reason', ''),
            'total_params': total_params,
            'trainable_params': trainable_params,
            'best_epoch': best_epoch,
            f'best_{CONFIG["selection_metric"]}': float(best_score),
            **{f'val_{k}': v for k, v in val_summary.items()},
            **{f'test_{k}': v for k, v in test_summary.items()},
            'checkpoint_path': str(ckpt_path),
        }
        all_fold_summaries.append(summary_row)
        pd.DataFrame([summary_row]).to_csv(model_dir / 'fold_reports' / f'fold{fold}_summary.csv', index=False)
        print(f"Fold {fold} done | val_macro_f1={val_summary['macro_f1']:.4f} test_macro_f1={test_summary['macro_f1']:.4f}")

summary_df = pd.DataFrame(all_fold_summaries)
summary_df.to_csv(OUTPUT_ROOT / 'localization_no_gan_model_comparison_summary_per_fold.csv', index=False)

completed_summary = summary_df[summary_df.get('status', '') == 'completed'].copy() if 'status' in summary_df.columns else summary_df.copy()
if not completed_summary.empty:
    aggregate_df = (
        completed_summary
        .groupby('model_name')
        .agg(
            folds=('fold', 'count'),
            val_macro_f1_mean=('val_macro_f1', 'mean'),
            val_macro_f1_std=('val_macro_f1', 'std'),
            test_macro_f1_mean=('test_macro_f1', 'mean'),
            test_macro_f1_std=('test_macro_f1', 'std'),
            test_balanced_accuracy_mean=('test_balanced_accuracy', 'mean'),
            test_balanced_accuracy_std=('test_balanced_accuracy', 'std'),
            test_accuracy_mean=('test_accuracy', 'mean'),
            test_accuracy_std=('test_accuracy', 'std'),
        )
        .reset_index()
    )
else:
    aggregate_df = pd.DataFrame()
aggregate_df.to_csv(OUTPUT_ROOT / 'localization_no_gan_model_comparison_aggregate.csv', index=False)

with pd.ExcelWriter(OUTPUT_ROOT / 'localization_no_gan_model_comparison_crossval_summary.xlsx', engine='openpyxl') as writer:
    summary_df.to_excel(writer, sheet_name='summary_per_fold', index=False)
    aggregate_df.to_excel(writer, sheet_name='aggregate', index=False)
    model_availability_df.to_excel(writer, sheet_name='model_availability', index=False)
    fold_distribution_df.to_excel(writer, sheet_name='fold_distribution', index=False)
    for key, hist in list(all_histories.items())[:20]:
        sheet = key[:31]
        hist.to_excel(writer, sheet_name=sheet, index=False)

print('Saved summary:', OUTPUT_ROOT / 'localization_no_gan_model_comparison_summary_per_fold.csv')
print('Saved aggregate:', OUTPUT_ROOT / 'localization_no_gan_model_comparison_aggregate.csv')
print('Saved Excel  :', OUTPUT_ROOT / 'localization_no_gan_model_comparison_crossval_summary.xlsx')
display(summary_df)
display(aggregate_df)


/tmp/ipykernel_16750/330691398.py:6: RuntimeWarning: Minimum class count in CV data is 2, smaller than n_splits=5. Using rare-class tolerant round-robin folds; some validation folds may not contain every class.
  warnings.warn(


,fold,NORM,IMI,AMI,LMI
0,1,13,13,7,1
1,2,13,13,7,1
2,3,13,13,7,0
3,4,13,13,6,0
4,5,12,12,6,0



MODEL: cnn_lstm
OUTPUT: /home/nugee/code-program/code-thesis/hibah/myocardial-infarction-classification/crossval_comparison/localization_cnn_lstm
Fold 1/5 | train=129 val=34 | pretrained=True | trainable=117,700/117,700
Epoch 001 | train_loss=12.2544 val_loss=6.9004 val_acc=0.7941 val_macro_f1=0.6107
Epoch 005 | train_loss=0.6921 val_loss=5.1024 val_acc=0.7059 val_macro_f1=0.5625
Epoch 010 | train_loss=0.4765 val_loss=2.9540 val_acc=0.8235 val_macro_f1=0.6306
Epoch 015 | train_loss=0.5160 val_loss=3.8586 val_acc=0.7941 val_macro_f1=0.6200
Epoch 020 | train_loss=0.2790 val_loss=7.0847 val_acc=0.7941 val_macro_f1=0.6115
Epoch 025 | train_loss=0.3980 val_loss=4.1117 val_acc=0.8235 val_macro_f1=0.6374
Epoch 030 | train_loss=0.3344 val_loss=9.5082 val_acc=0.8235 val_macro_f1=0.6222
Epoch 035 | train_loss=0.3929 val_loss=5.2821 val_acc=0.8235 val_macro_f1=0.6198
Epoch 040 | train_loss=0.2955 val_loss=8.9626 val_acc=0.7941 val_macro_f1=0.6099
Epoch 045 | train_loss=0.3479 val_loss=4.9963 val

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Fold 1/5 | train=129 val=34 | pretrained=True | trainable=93,126,788/93,126,788
Epoch 001 | train_loss=0.8959 val_loss=1.1372 val_acc=0.0294 val_macro_f1=0.0143
Epoch 005 | train_loss=0.2105 val_loss=0.7627 val_acc=0.1471 val_macro_f1=0.1094
Epoch 010 | train_loss=0.0558 val_loss=0.7416 val_acc=0.7059 val_macro_f1=0.7710
Epoch 015 | train_loss=0.0428 val_loss=0.6949 val_acc=0.7941 val_macro_f1=0.8390
Epoch 020 | train_loss=0.0316 val_loss=0.9298 val_acc=0.7353 val_macro_f1=0.5592
Epoch 025 | train_loss=0.0224 val_loss=1.1042 val_acc=0.6471 val_macro_f1=0.5076
Epoch 030 | train_loss=0.0231 val_loss=1.2401 val_acc=0.6765 val_macro_f1=0.5275
Epoch 035 | train_loss=0.0286 val_loss=1.3316 val_acc=0.7059 val_macro_f1=0.5541
Epoch 040 | train_loss=0.0223 val_loss=1.3414 val_acc=0.7059 val_macro_f1=0.5541
Epoch 045 | train_loss=0.0156 val_loss=1.3643 val_acc=0.7059 val_macro_f1=0.5541
Epoch 050 | train_loss=0.0315 val_loss=1.3749 val_acc=0.7059 val_macro_f1=0.5541
Epoch 055 | train_loss=0.0147

...

Fold 4/5 | train=131 val=32 | pretrained=True | trainable=93,126,788/93,126,788
Epoch 001 | train_loss=1.1038 val_loss=2.4910 val_acc=0.0000 val_macro_f1=0.0000
Epoch 005 | train_loss=0.2762 val_loss=2.0100 val_acc=0.1875 val_macro_f1=0.2255
Epoch 010 | train_loss=0.0980 val_loss=1.2197 val_acc=0.5000 val_macro_f1=0.3962
Epoch 015 | train_loss=0.0660 val_loss=1.3500 val_acc=0.5312 val_macro_f1=0.4029
Epoch 020 | train_loss=0.0793 val_loss=1.4553 val_acc=0.5000 val_macro_f1=0.3759
Epoch 025 | train_loss=0.0537 val_loss=1.4612 val_acc=0.5312 val_macro_f1=0.4111
Epoch 030 | train_loss=0.0643 val_loss=1.3892 val_acc=0.5938 val_macro_f1=0.4942
Epoch 035 | train_loss=0.0394 val_loss=1.2971 val_acc=0.5000 val_macro_f1=0.3936
Epoch 040 | train_loss=0.0548 val_loss=1.3396 val_acc=0.5312 val_macro_f1=0.4322
Epoch 045 | train_loss=0.0374 val_loss=1.3389 val_acc=0.5625 val_macro_f1=0.4607
Epoch 050 | train_loss=0.0389 val_loss=1.4655 val_acc=0.6250 val_macro_f1=0.4908
Epoch 055 | train_loss=0.0445

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

Fold 5/5 | train=133 val=30 | pretrained=True | trainable=93,126,788/93,126,788
Epoch 001 | train_loss=1.1170 val_loss=2.1275 val_acc=0.0000 val_macro_f1=0.0000
Epoch 005 | train_loss=0.3580 val_loss=1.4558 val_acc=0.2333 val_macro_f1=0.2665
Epoch 010 | train_loss=0.1056 val_loss=0.8127 val_acc=0.6333 val_macro_f1=0.4978
Epoch 015 | train_loss=0.1013 val_loss=0.8297 val_acc=0.5667 val_macro_f1=0.5833
Epoch 020 | train_loss=0.0469 val_loss=1.0204 val_acc=0.6000 val_macro_f1=0.6000
Epoch 025 | train_loss=0.0400 val_loss=1.1243 val_acc=0.5333 val_macro_f1=0.4208
Epoch 030 | train_loss=0.0297 val_loss=1.0763 val_acc=0.5667 val_macro_f1=0.4343
Epoch 035 | train_loss=0.0780 val_loss=1.1369 val_acc=0.5333 val_macro_f1=0.4150
Epoch 040 | train_loss=0.0345 val_loss=1.1612 val_acc=0.5333 val_macro_f1=0.4150
Epoch 045 | train_loss=0.0408 val_loss=1.1820 val_acc=0.5000 val_macro_f1=0.3950
Epoch 050 | train_loss=0.0307 val_loss=1.1908 val_acc=0.5000 val_macro_f1=0.3950
Epoch 055 | train_loss=0.0329

,model_name,fold,status,finetuning_mode,pretrain_loaded,pretrain_reason,total_params,trainable_params,best_epoch,best_macro_f1,...,val_accuracy,val_balanced_accuracy,val_macro_f1,val_weighted_f1,test_loss,test_accuracy,test_balanced_accuracy,test_macro_f1,test_weighted_f1,checkpoint_path
0,cnn_lstm,1,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,117700,117700,36,0.666429,...,0.882353,0.675824,0.666429,0.867983,0.516875,0.880952,0.909722,0.914413,0.879385,/home/nugee/code-program/code-thesis/hibah/myo...
1,cnn_lstm,2,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,117700,117700,3,0.598214,...,0.764706,0.598901,0.598214,0.763655,5.638476,0.738095,0.557292,0.560577,0.729212,/home/nugee/code-program/code-thesis/hibah/myo...
2,cnn_lstm,3,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,117700,117700,1,0.871605,...,0.848485,0.871795,0.871605,0.848260,3.222995,0.785714,0.612847,0.610416,0.773295,/home/nugee/code-program/code-thesis/hibah/myo...
3,cnn_lstm,4,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,117700,117700,99,0.881766,...,0.875000,0.897436,0.881766,0.872730,0.861884,0.833333,0.878472,0.772331,0.841451,/home/nugee/code-program/code-thesis/hibah/myo...
4,cnn_lstm,5,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,117700,117700,26,0.847475,...,0.833333,0.833333,0.847475,0.835152,0.464786,0.809524,0.862847,0.721380,0.832051,/home/nugee/code-program/code-thesis/hibah/myo...
5,cnn1d,1,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,10260,10260,1,0.638440,...,0.823529,0.637363,0.638440,0.813541,2.551704,0.714286,0.529514,0.537695,0.700297,/home/nugee/code-program/code-thesis/hibah/myo...
6,cnn1d,2,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,10260,10260,1,0.585371,...,0.764706,0.582418,0.585371,0.748215,2.846597,0.761905,0.585069,0.586949,0.748565,/home/nugee/code-program/code-thesis/hibah/myo...
7,cnn1d,3,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,10260,10260,1,0.847203,...,0.848485,0.827839,0.847203,0.849724,2.057582,0.738095,0.557292,0.563553,0.725414,/home/nugee/code-program/code-thesis/hibah/myo...
8,cnn1d,4,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,10260,10260,1,0.810227,...,0.812500,0.786325,0.810227,0.812464,2.003041,0.761905,0.585069,0.586949,0.748565,/home/nugee/code-program/code-thesis/hibah/myo...
9,cnn1d,5,completed,full,True,/home/nugee/code-program/code-thesis/hibah/myo...,10260,10260,1,0.835097,...,0.833333,0.833333,0.835097,0.835450,1.870304,0.761905,0.585069,0.586949,0.748565,/home/nugee/code-program/code-thesis/hibah/myo...


,model_name,folds,val_macro_f1_mean,val_macro_f1_std,test_macro_f1_mean,test_macro_f1_std,test_balanced_accuracy_mean,test_balanced_accuracy_std,test_accuracy_mean,test_accuracy_std
0,GRU,5,0.781403,0.120119,0.666116,0.151737,0.753125,0.100847,0.728571,0.125537
1,cnn1d,5,0.743268,0.122105,0.572419,0.021896,0.568403,0.024845,0.747619,0.021296
2,cnn_lstm,5,0.773098,0.131346,0.715824,0.139530,0.764236,0.165594,0.809524,0.053240
3,ecg_fm,5,0.670884,0.083331,0.833360,0.022941,0.845833,0.022616,0.776190,0.027147
4,hubert_ecg,5,0.671058,0.114047,0.652422,0.085148,0.739236,0.110856,0.700000,0.054814
5,lstm,5,0.797780,0.124025,0.678558,0.150519,0.744792,0.175669,0.776190,0.097590


## 9. Output Structure

Setiap model menyimpan hasil ke folder:

- `crossval_comparison/localization_cnn_lstm/`
- `crossval_comparison/localization_cnn1d/`
- `crossval_comparison/localization_GRU/`
- `crossval_comparison/localization_lstm/`
- `crossval_comparison/localization_hubert_ecg/` jika tersedia
- `crossval_comparison/localization_ecg_fm/` jika tersedia

Di setiap folder model tersedia struktur yang sama dengan notebook transfer learning crossval sebelumnya:

- `fold_models/`
- `fold_histories/`
- `fold_plots/`
- `fold_reports/`

Ringkasan lintas model disimpan di root `crossval_comparison/`:

- `localization_no_gan_model_comparison_summary_per_fold.csv`
- `localization_no_gan_model_comparison_aggregate.csv`
- `localization_no_gan_model_comparison_crossval_summary.xlsx`
- `localization_model_availability.csv`

Catatan: HuBERT ECG dan ECG-FM sengaja dibuat opsional karena dependency/model resmi dapat berbeda antar environment. HuBERT ECG sekarang diarahkan ke folder lokal `model_pretrain_comparison/hubert_ecg`; ubah `CONFIG['hubert_ecg_model_id_or_path']` hanya bila folder model dipindahkan.
